In [1]:
from sklearn.ensemble import RandomForestRegressor
import os
import numpy as np
import pandas as pd
import sys
sys.path.append(os.path.dirname(os.getcwd()))
import functions

In [2]:
pd.__version__

'3.0.3'

Data wrangling for the input data to models

In [3]:
# df = pd.read_csv('../data/features_table.csv', 
#                  parse_dates=['order_date'])
df = pd.read_parquet('../data/features_table.parquet')

# add an order_number to each row in df, in order of the order_date
df['order_number'] = df['order_date'].rank(method='first').astype(int)

# Do one-hot encoding first, so that the train and test columns match
features_data = functions.one_hot_encode(df, ['origin_country', 'us_destination_state'])

In [4]:
df.dtypes

origin_country                     str
us_destination_state               str
projected_days                   int64
actual_days                      int64
order_date              datetime64[us]
order_number                     int64
dtype: object

In [5]:
# Choose the run_date for the train-test split
run_date = features_data['order_date'].quantile(0.9)
print(run_date)
# The window of dates will be one week
print(run_date + pd.Timedelta(days=7))

2017-12-26 00:00:00
2018-01-02 00:00:00


In [6]:
# Training data to include all data up to the run_date, 
# and testing data to include all data from the run_date to one week after the run_date
features_train = features_data[features_data['order_date'] < run_date]
features_test1 = features_data[features_data['order_date'] >= run_date]
features_test = features_test1[features_test1['order_date'] <= run_date + pd.Timedelta(days=7)]

# Remove order date from features, maybe add back for future features, such as a rolling average or seasonality
features_train = features_train.drop(columns=['order_date', 'order_number'])
features_test = features_test.drop(columns=['order_date', 'order_number'])

features_train.reset_index(drop=True, inplace=True)
features_test.reset_index(drop=True, inplace=True)

# drop non-feature columns, and move the target actual_days to the last columns
feature_cols = [col for col in features_train.columns if col != 'actual_days']
features_train = features_train[feature_cols + ['actual_days']]
features_train.reset_index(drop=True, inplace=True)

print(len(features_train), len(features_test))

680 19


In [7]:
# Create training and testing variables
X_train = features_train[feature_cols]
y_train = features_train['actual_days']
X_test = features_test[feature_cols]
y_test = features_test['actual_days']

Random Forest Model
* Machine learning model that utilizes decision trees

In [8]:
# Simple Random Forest regressor
model = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

In [9]:
rf_data = (
    df.loc[
        (df["order_date"] >= run_date)
        & (df["order_date"] <= run_date + pd.Timedelta(days=7)),
        ["origin_country", "us_destination_state", "order_date", "order_number"],
    ]
    .reset_index(drop=True)
    .copy()
)

In [10]:
rf_results = functions.summarize_results(rf_data, run_date, y_test, y_pred, model_name='Random Forest')
print('Unique Origin-Destination pairs:',len(rf_results[['origin_country', 'us_destination_state']].drop_duplicates()))
rf_results

Predicted rows: 19
 MAE: 1.043 
 RMSE: 1.257 
 R2: 0.000 
 MAPE: 0.307
Unique Origin-Destination pairs: 9


,origin_country,us_destination_state,order_date,order_number,actual,pred,abs_err,model
0,Vietnam,PR,2017-12-26,681,4,4.18,0.18,Random Forest
1,Vietnam,PR,2017-12-26,682,3,4.18,1.18,Random Forest
2,Vietnam,CA,2017-12-26,683,2,3.87,1.87,Random Forest
3,Vietnam,PR,2017-12-27,684,4,4.18,0.18,Random Forest
4,Vietnam,PR,2017-12-27,685,3,4.61,1.61,Random Forest
5,Vietnam,OH,2017-12-28,686,6,4.80,1.20,Random Forest
6,Vietnam,MD,2017-12-28,687,4,2.93,1.07,Random Forest
7,Vietnam,OH,2017-12-28,688,4,3.56,0.44,Random Forest
8,Vietnam,OH,2017-12-29,689,5,4.80,0.20,Random Forest
9,Vietnam,MO,2017-12-29,690,5,6.00,1.00,Random Forest


In [11]:
# How much weight did the model put on each feature?
feature_importances = pd.DataFrame({'feature': feature_cols, 'importance': model.feature_importances_}).sort_values('importance', ascending=False)
feature_importances

# How many observations were in the training set for each feature?
feature_counts = pd.DataFrame({'feature': feature_cols, 'count': X_train[feature_cols].sum()}).sort_values('count', ascending=False)
feature_counts

# Merge feature importance and counts to see if there's a relationship between them
feature_analysis = pd.merge(feature_importances, feature_counts, on='feature')
feature_analysis.sort_values('importance', ascending=False)

,feature,importance,count
0,projected_days,0.676386,2117
1,us_destination_state_FL,0.029902,25
2,us_destination_state_IL,0.026057,42
3,us_destination_state_MO,0.023831,6
4,us_destination_state_TX,0.023339,36
5,us_destination_state_SC,0.020695,6
6,us_destination_state_DC,0.019879,9
7,us_destination_state_MD,0.019823,8
8,us_destination_state_PR,0.017408,234
9,us_destination_state_MI,0.017067,22


Naive model
* Uses the latest observation to predict the next (week's) observation(s)

In [12]:
# Training data to include all data up to the run_date, 
# and testing data to include all data from the run_date to one week after the run_date
# Unlike the machine learning model, there is no one-hot encoding and the order_date is retained
a_train1 = df[df['order_date'] < run_date]
a_test1 = df[df['order_date'] >= run_date]
a_test = a_test1[a_test1['order_date'] <= run_date + pd.Timedelta(days=7)]

In [13]:
# Filter a_train to only include the latest order_date for each origin_country and us_destination_state combination
a_train = a_train1.sort_values('order_date').groupby(['origin_country', 'us_destination_state']).tail(1).reset_index(drop=True)
# a_train.sort_values('order_date')
a_train.sort_values('us_destination_state')

,origin_country,us_destination_state,projected_days,actual_days,order_date,order_number
12,Vietnam,AR,4,4,2016-02-19,470
23,Vietnam,AZ,1,2,2017-12-05,655
30,Vietnam,CA,1,2,2017-12-25,676
16,Vietnam,CO,4,5,2016-03-08,559
4,Vietnam,CT,1,2,2016-01-12,244
21,Vietnam,DC,1,2,2017-11-20,647
26,Vietnam,FL,4,6,2017-12-13,668
13,Vietnam,GA,4,4,2016-02-21,482
5,Vietnam,HI,4,4,2016-01-17,272
1,Vietnam,ID,1,2,2015-10-31,21


In [14]:
# # To verify logic of choosing the latest order_date in the training set, could alternatively write a unit test
# print(run_date)
# # df[df['us_destination_state']=='AR']
# df[df['us_destination_state']=='AZ'].sort_values('order_date')

In [15]:
actuals = a_test[['origin_country', 'us_destination_state', 'order_date', 'order_number', 'actual_days']].reset_index(drop=True)
actuals

,origin_country,us_destination_state,order_date,order_number,actual_days
0,Vietnam,PR,2017-12-26,681,4
1,Vietnam,PR,2017-12-26,682,3
2,Vietnam,CA,2017-12-26,683,2
3,Vietnam,PR,2017-12-27,684,4
4,Vietnam,PR,2017-12-27,685,3
5,Vietnam,OH,2017-12-28,686,6
6,Vietnam,MD,2017-12-28,687,4
7,Vietnam,OH,2017-12-28,688,4
8,Vietnam,OH,2017-12-29,689,5
9,Vietnam,MO,2017-12-29,690,5


In [16]:
# add the predicted_days columne to naive, and fill it with the actual_days for each origin_country and us_destination_state combination in a_train
naive = actuals.copy()
naive['predicted_days'] = naive.apply(lambda row: a_train[(a_train['origin_country'] == row['origin_country']) & (a_train['us_destination_state'] == row['us_destination_state'])]['actual_days'].iloc[-1] if not a_train[(a_train['origin_country'] == row['origin_country']) & (a_train['us_destination_state'] == row['us_destination_state'])].empty else None, axis=1)

# if naive['predicted_days'] is null, remove those rows from the data, we can handle those with a different method (master table) later
naive = naive.dropna(subset=['predicted_days']).reset_index(drop=True)
# naive['predicted_days'] = naive['predicted_days'].fillna(a_train['actual_days'].mean())
naive

,origin_country,us_destination_state,order_date,order_number,actual_days,predicted_days
0,Vietnam,PR,2017-12-26,681,4,6
1,Vietnam,PR,2017-12-26,682,3,6
2,Vietnam,CA,2017-12-26,683,2,2
3,Vietnam,PR,2017-12-27,684,4,6
4,Vietnam,PR,2017-12-27,685,3,6
5,Vietnam,OH,2017-12-28,686,6,6
6,Vietnam,MD,2017-12-28,687,4,4
7,Vietnam,OH,2017-12-28,688,4,6
8,Vietnam,OH,2017-12-29,689,5,6
9,Vietnam,MO,2017-12-29,690,5,6


In [17]:
# Check for nan values in the predicted_days column, can be moved to a unit test later
print(naive['predicted_days'].isna().sum())
print(naive['actual_days'].unique())
print(naive['predicted_days'].unique())

0
[4 3 2 6 5]
[6 2 4 5]


In [18]:
naive

,origin_country,us_destination_state,order_date,order_number,actual_days,predicted_days
0,Vietnam,PR,2017-12-26,681,4,6
1,Vietnam,PR,2017-12-26,682,3,6
2,Vietnam,CA,2017-12-26,683,2,2
3,Vietnam,PR,2017-12-27,684,4,6
4,Vietnam,PR,2017-12-27,685,3,6
5,Vietnam,OH,2017-12-28,686,6,6
6,Vietnam,MD,2017-12-28,687,4,4
7,Vietnam,OH,2017-12-28,688,4,6
8,Vietnam,OH,2017-12-29,689,5,6
9,Vietnam,MO,2017-12-29,690,5,6


In [19]:
# Calculate summary metrics
n_results = functions.summarize_results(naive, 
                                        run_date, 
                                        y_test=naive['actual_days'], y_pred=naive['predicted_days'],
                                        model_name='Naive')
print('Unique Origin-Destination pairs:',len(n_results[['origin_country', 'us_destination_state']].drop_duplicates()))
n_results

Predicted rows: 19
 MAE: 1.526 
 RMSE: 1.933 
 R2: -1.367 
 MAPE: 0.496
Unique Origin-Destination pairs: 9


,origin_country,us_destination_state,order_date,order_number,actual,pred,abs_err,model
0,Vietnam,PR,2017-12-26,681,4,6,2,Naive
1,Vietnam,PR,2017-12-26,682,3,6,3,Naive
2,Vietnam,CA,2017-12-26,683,2,2,0,Naive
3,Vietnam,PR,2017-12-27,684,4,6,2,Naive
4,Vietnam,PR,2017-12-27,685,3,6,3,Naive
5,Vietnam,OH,2017-12-28,686,6,6,0,Naive
6,Vietnam,MD,2017-12-28,687,4,4,0,Naive
7,Vietnam,OH,2017-12-28,688,4,6,2,Naive
8,Vietnam,OH,2017-12-29,689,5,6,1,Naive
9,Vietnam,MO,2017-12-29,690,5,6,1,Naive


In [20]:
# Which rows in particular increased the MAE and RMSE the most? Show the top 10 rows with the highest absolute error between actual and predicted days, along with their origin_country and us_destination_state.
naive['abs_error'] = (naive['actual_days'] - naive['predicted_days']).abs()
naive.sort_values('abs_error', ascending=False)[['origin_country', 'us_destination_state', 'actual_days', 'predicted_days', 'abs_error']]

,origin_country,us_destination_state,actual_days,predicted_days,abs_error
13,Vietnam,PR,2,6,4
4,Vietnam,PR,3,6,3
1,Vietnam,PR,3,6,3
16,Vietnam,CO,2,5,3
0,Vietnam,PR,4,6,2
11,Vietnam,NJ,4,6,2
3,Vietnam,PR,4,6,2
7,Vietnam,OH,4,6,2
15,Vietnam,PR,4,6,2
18,Vietnam,FL,4,6,2


In [21]:
# PR did have the most recent actual value of 6
df[df['us_destination_state'] == 'PR'].sort_values('order_date', ascending=False)[['origin_country', 'us_destination_state', 'order_date', 'actual_days']]

,origin_country,us_destination_state,order_date,actual_days
122671,Vietnam,PR,2018-02-06,6
124255,Vietnam,PR,2018-02-05,6
125493,Vietnam,PR,2018-02-02,6
150605,Vietnam,PR,2018-01-30,2
41405,Vietnam,PR,2018-01-29,2
...,...,...,...,...
67161,Vietnam,PR,2015-10-28,4
41641,Vietnam,PR,2015-10-28,4
102780,Vietnam,PR,2015-10-28,4
67140,Vietnam,PR,2015-10-28,4


Master Table (versus simply naive method)
* Median of the last four weeks
* And fill in with earlier Median if the last four weeks has no data

In [22]:
# Show the last four weeks of history leading up to the run date, sorted by order_date
history = df[
    (df['order_date'] >= run_date - pd.Timedelta(weeks=4)) &
    (df['order_date'] < run_date)
].sort_values('order_date')
history

,origin_country,us_destination_state,projected_days,actual_days,order_date,order_number
97817,Vietnam,IL,2,3,2017-12-01,652
160399,Vietnam,PR,4,3,2017-12-02,653
169463,Vietnam,NC,2,6,2017-12-04,654
1223,Vietnam,AZ,1,2,2017-12-05,655
1222,Vietnam,PR,4,3,2017-12-06,656
1226,Vietnam,PR,4,4,2017-12-07,657
1230,Vietnam,FL,4,5,2017-12-08,658
1225,Vietnam,CA,4,5,2017-12-08,659
1229,Vietnam,OH,4,6,2017-12-09,660
1224,Vietnam,CA,4,6,2017-12-09,661


In [23]:
# Keep the median actual_days for each origin_country and us_destination_state combination in the last four weeks of history, and use that as the predicted_days for any rows in the test set that don't have a predicted_days value from the naive model (because they didn't have a matching origin_country and us_destination_state combination in the training set).
master_table = history.groupby(['origin_country', 'us_destination_state'])['actual_days'].median().reset_index()
master_table.rename(columns={'actual_days': 'median_actual_days'}, inplace=True)
master_table

,origin_country,us_destination_state,median_actual_days
0,Vietnam,AZ,2.0
1,Vietnam,CA,5.0
2,Vietnam,FL,5.5
3,Vietnam,IL,2.5
4,Vietnam,MI,3.0
5,Vietnam,NC,6.0
6,Vietnam,OH,6.0
7,Vietnam,PA,2.5
8,Vietnam,PR,3.0
9,Vietnam,TX,2.5


In [24]:
actuals

,origin_country,us_destination_state,order_date,order_number,actual_days
0,Vietnam,PR,2017-12-26,681,4
1,Vietnam,PR,2017-12-26,682,3
2,Vietnam,CA,2017-12-26,683,2
3,Vietnam,PR,2017-12-27,684,4
4,Vietnam,PR,2017-12-27,685,3
5,Vietnam,OH,2017-12-28,686,6
6,Vietnam,MD,2017-12-28,687,4
7,Vietnam,OH,2017-12-28,688,4
8,Vietnam,OH,2017-12-29,689,5
9,Vietnam,MO,2017-12-29,690,5


In [25]:
# Add the median_actual_days from the master_table to the actuals DataFrame as predicted_days, 
# matching on origin_country and us_destination_state. If there is no match in the master_table for a 
# given origin_country and us_destination_state combination, then predicted_days should be set to None.
md = actuals.merge(master_table, how='left', on=['origin_country', 'us_destination_state'])
md = md.rename(columns={'median_actual_days': 'predicted_days'})
md['predicted_days'] = md['predicted_days'].where(md['predicted_days'].notna(), None)
md


,origin_country,us_destination_state,order_date,order_number,actual_days,predicted_days
0,Vietnam,PR,2017-12-26,681,4,3.0
1,Vietnam,PR,2017-12-26,682,3,3.0
2,Vietnam,CA,2017-12-26,683,2,5.0
3,Vietnam,PR,2017-12-27,684,4,3.0
4,Vietnam,PR,2017-12-27,685,3,3.0
5,Vietnam,OH,2017-12-28,686,6,6.0
6,Vietnam,MD,2017-12-28,687,4,NaN
7,Vietnam,OH,2017-12-28,688,4,6.0
8,Vietnam,OH,2017-12-29,689,5,6.0
9,Vietnam,MO,2017-12-29,690,5,NaN


In [26]:
# Add in additional rows, from previous runs, for missing origin_country and us_destination_state combinations in the test set

# But for now, just don't predict for those missing combinations
# add the predicted_days columne to naive, and fill it with the actual_days for each origin_country and us_destination_state combination in a_train
md = md.dropna().reset_index(drop=True)
md

,origin_country,us_destination_state,order_date,order_number,actual_days,predicted_days
0,Vietnam,PR,2017-12-26,681,4,3.0
1,Vietnam,PR,2017-12-26,682,3,3.0
2,Vietnam,CA,2017-12-26,683,2,5.0
3,Vietnam,PR,2017-12-27,684,4,3.0
4,Vietnam,PR,2017-12-27,685,3,3.0
5,Vietnam,OH,2017-12-28,686,6,6.0
6,Vietnam,OH,2017-12-28,688,4,6.0
7,Vietnam,OH,2017-12-29,689,5,6.0
8,Vietnam,PR,2017-12-30,691,6,3.0
9,Vietnam,PR,2017-12-30,693,6,3.0


In [28]:
# Calculate summary metrics
md_results = functions.summarize_results(md, 
                                         run_date, 
                                         y_test=md['actual_days'], y_pred=md['predicted_days'],
                                         model_name='Master Table')
print('Unique Origin-Destination pairs:',len(md_results[['origin_country', 'us_destination_state']].drop_duplicates()))
md_results

Predicted rows: 14
 MAE: 1.393 
 RMSE: 1.737 
 R2: -0.782 
 MAPE: 0.373
Unique Origin-Destination pairs: 4


,origin_country,us_destination_state,order_date,order_number,actual,pred,abs_err,model
0,Vietnam,PR,2017-12-26,681,4,3.0,1.0,Master Table
1,Vietnam,PR,2017-12-26,682,3,3.0,0.0,Master Table
2,Vietnam,CA,2017-12-26,683,2,5.0,3.0,Master Table
3,Vietnam,PR,2017-12-27,684,4,3.0,1.0,Master Table
4,Vietnam,PR,2017-12-27,685,3,3.0,0.0,Master Table
5,Vietnam,OH,2017-12-28,686,6,6.0,0.0,Master Table
6,Vietnam,OH,2017-12-28,688,4,6.0,2.0,Master Table
7,Vietnam,OH,2017-12-29,689,5,6.0,1.0,Master Table
8,Vietnam,PR,2017-12-30,691,6,3.0,3.0,Master Table
9,Vietnam,PR,2017-12-30,693,6,3.0,3.0,Master Table


Summary

In [30]:
# append rf_results, n_results, and md_results into a new summary table
rf_summary = rf_results.rename(columns={'actual': 'actual_days', 'pred': 'predicted_days'}).drop(columns=['abs_err'])
summary = pd.concat([rf_summary, n_results, md_results], ignore_index=True, sort=False)
summary = summary[['origin_country', 'us_destination_state', 'actual_days', 'predicted_days', 'model', 'order_date', 'order_number']]
summary

,origin_country,us_destination_state,actual_days,predicted_days,model,order_date,order_number
0,Vietnam,PR,4.0,4.18,Random Forest,2017-12-26,681
1,Vietnam,PR,3.0,4.18,Random Forest,2017-12-26,682
2,Vietnam,CA,2.0,3.87,Random Forest,2017-12-26,683
3,Vietnam,PR,4.0,4.18,Random Forest,2017-12-27,684
4,Vietnam,PR,3.0,4.61,Random Forest,2017-12-27,685
5,Vietnam,OH,6.0,4.80,Random Forest,2017-12-28,686
6,Vietnam,MD,4.0,2.93,Random Forest,2017-12-28,687
7,Vietnam,OH,4.0,3.56,Random Forest,2017-12-28,688
8,Vietnam,OH,5.0,4.80,Random Forest,2017-12-29,689
9,Vietnam,MO,5.0,6.00,Random Forest,2017-12-29,690


In [31]:
# How many observations were in the test set for each model?
summary.groupby('model').count()['actual_days']

model
Master Table      0
Naive             0
Random Forest    19
Name: actual_days, dtype: int64

In [32]:
# Using the df summary, pivot the "model" column such athat each of the values of "model" becomes a separate column, and the values in those columns are the predicted_days. The actual_days column should stay the same.
summary_pivot = summary.pivot_table(index=['origin_country', 'us_destination_state', 'order_date', 'order_number', 'actual_days',]
                                    , columns='model'
                                    , values='predicted_days').reset_index()
summary_pivot

model,origin_country,us_destination_state,order_date,order_number,actual_days,Random Forest
0,Vietnam,CA,2017-12-26,683,2.0,3.87
1,Vietnam,CO,2018-01-02,697,2.0,2.00
2,Vietnam,FL,2018-01-02,699,4.0,5.49
3,Vietnam,MD,2017-12-28,687,4.0,2.93
4,Vietnam,MO,2017-12-29,690,5.0,6.00
5,Vietnam,NJ,2017-12-30,692,4.0,5.32
6,Vietnam,OH,2017-12-28,686,6.0,4.80
7,Vietnam,OH,2017-12-28,688,4.0,3.56
8,Vietnam,OH,2017-12-29,689,5.0,4.80
9,Vietnam,PR,2017-12-26,681,4.0,4.18
